# SMS Spam Classification

This notebook builds a text classification pipeline for detecting spam messages using the SMS Spam Collection dataset.



### Project Overview

1. Download the SMS Spam Collection Dataset from the URL  
   https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip  
   Unzip this file to create a file named `SMSSpamCollection`.

2. Read this file and create a dataset with the following steps:

    - Create a Pandas DataFrame `df`:

    ```python
    df = pd.read_csv("SMSSpamCollection", sep='\t', header=None, names=['label', 'message'])
    ```

    - Then randomly pick 500 ham(non-spam) and 500 spam messages:
    ```python
    df = pd.concat([
        df[df['label'] == 'ham'].sample(n=500, random_state=42),
        df[df['label'] == 'spam'].sample(n=500, random_state=42)
    ])
    ```
    
    - Next, split `df` into train and test sets:
    ```python
    X_train, X_test, y_train, y_test = train_test_split(
        df['message'], df['label'], test_size=0.25, random_state=42, stratify=df['label']
    )
    ```

3. Try various vectorization methods and word normalization for the documents (SNS messages):

   - CountVectorizer, TfidfVectorizer
   - 1-gram, 2-gram, 3-gram

4. Use a Logistic Regression Classifier. No parameter tuning or cross validation is needed.

5. Find the best combination of vectorizer and n-gram in terms of the F1 score for detecting spam.

6. Explain your approach and discuss what was difficult and interesting (at least 5 lines).

7. Submit your program and the execution results in Jupyter Notebook format.

8. In addition to the above, please complete a survey given on Manaba-R.


In [ ]:
import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
urllib.request.urlretrieve(url, "smsspamcollection.zip")

with zipfile.ZipFile("smsspamcollection.zip", 'r') as zip_ref:
    zip_ref.extractall()

assert os.path.exists("SMSSpamCollection"), "SMSSpamCollection file not found!"
print("Dataset ready.")


Dataset ready.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings("ignore")


In [ ]:
df = pd.read_csv("SMSSpamCollection", sep='\t', header=None, names=['label', 'message'])

df = pd.concat([
    df[df['label'] == 'ham'].sample(n=500, random_state=42),
    df[df['label'] == 'spam'].sample(n=500, random_state=42)
])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
  df['message'], df['label'], test_size=0.25, random_state=42, stratify=df['label']
)


In [ ]:
vectorizers = {
    'CountVectorizer': CountVectorizer,
    'TfidfVectorizer': TfidfVectorizer
}

n_grams = [(1,1), (1,2), (1,3)]

results = []

for vec_name, VecClass in vectorizers.items():
    for ngram in n_grams:
        vectorizer = VecClass(ngram_range=ngram)
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_train_vec, y_train)
        y_pred = clf.predict(X_test_vec)

        score = f1_score(y_test, y_pred, pos_label='spam')
        results.append((vec_name, ngram, score))
        print(f"{vec_name} {ngram}: F1 = {score:.4f}")


CountVectorizer (1, 1): F1 = 0.9461
CountVectorizer (1, 2): F1 = 0.9372
CountVectorizer (1, 3): F1 = 0.9283
TfidfVectorizer (1, 1): F1 = 0.9444
TfidfVectorizer (1, 2): F1 = 0.9486
TfidfVectorizer (1, 3): F1 = 0.9486


In [ ]:
best = max(results, key=lambda x: x[2])
print(f"\nBest F1 Score: {best[2]:.4f} using {best[0]} with n-gram {best[1]}")


Best F1 Score: 0.9486 using TfidfVectorizer with n-gram (1, 2)


I began by balancing the dataset with 500 spam and 500 ham messages, followed by stratified train-test split so that both classes were equally represented in both sets.

I tested two vectorization techniques—CountVectorizer and TfidfVectorizer—with 1-gram, 2-gram, and 3-gram settings to convert messages into numerical features.

I used a simple Logistic Regression classifier trained on each of the vectorized forms without performing any hyperparameter tuning.

The optimal F1 score (0.9486) was achieved with TfidfVectorizer using n-gram ranges (1,2) and (1,3), which shows TF-IDF and bi/tri-grams pick up common spam trends well.

Surprisingly, the performance of CountVectorizer weakened only slightly as the size of the n-gram range increased, likely because there is sparsity in larger-order n-grams.

It was interesting to observe how feature engineering—n-gram modeling in this case—is so beneficial to model performance even in a simple classifier.

The hardest part was trying many configurations efficiently and understanding how text representation changes in small increments affect classification accuracy.
